In [1]:
import scanpy as sc 
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA

import scipy.sparse as sp 
import warnings

warnings.filterwarnings("ignore")

import os
import ctypes
import sys

# 1. 先设置 R_HOME
os.environ["R_HOME"] = "/home/pxy/miniconda3/envs/r40/lib/R"

# 2. 【核心黑科技】手动加载 R 的动态库
# 这步操作等同于在终端里设置 LD_LIBRARY_PATH，专门解决 VS Code 找不到库的问题
try:
    # 这是 R 的核心库路径
    libR_path = "/home/pxy/miniconda3/envs/r40/lib/R/lib/libR.so"
    # 强制加载进内存
    ctypes.CDLL(libR_path, mode=ctypes.RTLD_GLOBAL)
    print("✅ 成功强制加载 libR.so")
except OSError as e:
    print(f"❌ 加载失败: {e}")

# 3. 然后再导入其他包
sys.path.append("..") 

import spCLUE
import rpy2.robjects as robjects
print("R 环境路径:", robjects.r['R.home']()[0])

spCLUE.fix_seed(0)

# 定义DLPFC数据集的12个切片ID
slice_ids = [
    "151507", "151508", "151509", "151510",
    "151669", "151670", "151671", "151672",
    "151673", "151674", "151675", "151676"
]

# 用于存储每个切片的ARI结果
ari_results = []

# 数据路径（请根据实际情况确认路径是否正确）
data_dir = '/home/pxy/home/pxy/data/DLPFC/st/'

# 【新增】创建保存图片的文件夹
figures_dir = "figures_test"
if not os.path.exists(figures_dir):
    os.makedirs(figures_dir)
    print(f"Created directory: {figures_dir}")

print(f"Start processing {len(slice_ids)} slices...")

for sample_name in slice_ids:
    print(f"\n{'='*20} Processing Sample: {sample_name} {'='*20}")
    
    # 1. 设置簇的数量 (根据DLPFC数据集的已知Ground Truth)
    # 151669-151672 通常只有5层，其他切片为7层
    if sample_name in ["151669", "151670", "151671", "151672"]:
        n_clusters = 5
    else:
        n_clusters = 7
    
    try:
        # 2. 加载数据
        # 使用 read_visium 加载数据，路径拼接逻辑参考原文件
        adata = sc.read_visium(data_dir + sample_name)
        adata.var_names_make_unique()
        
        # 加载元数据 (Ground Truth)
        meta = pd.read_csv(data_dir + sample_name + "/metadata.tsv", sep="\t")
        meta = meta.set_index("barcode")
        adata.obs["Region"] = meta.loc[adata.obs_names, "layer_guess_reordered"]
        
        # 3. 数据预处理与构图
        # 原文件 Cell 6 的逻辑
        # adata = spCLUE.preprocess(adata)
        # adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X)
        
        # g_spatial = spCLUE.prepare_graph(adata, "spatial", n_neighbors=6)
        # g_expr = spCLUE.prepare_graph(adata, "expr", metric="euclidean", n_neighbors=8)
        from spCLUE.preprocess import (
            prepare_euclidean_graph, 
            prepare_cosine_graph, 
            prepare_fused_graph
        )

        # 1. 数据预处理
        adata = spCLUE.preprocess(adata, hvgNumber=3000)

        # 2. 构建三个视图的图
        adj_s, g_spatial = prepare_euclidean_graph(adata,)
        adj_f, g_feature = prepare_cosine_graph(adata, k=14)
        g_combined = prepare_fused_graph(adj_s, adj_f)

        # 3. 准备图字典
        graph_dict = {
            "spatial": g_spatial,
            "feature": g_feature,
            "combined": g_combined,
        }
        use_zinb = True
        
        # 4. 模型初始化与训练
        # 原文件 Cell 8 的逻辑
        # 注意：这里将 n_clusters 参数改为动态变量，与当前切片保持一致
        # spCLUE_model = spCLUE.spCLUE(adata.obsm["X_pca"], graph_dict, n_clusters,
                                    # )
        # _, adata.obsm["spCLUE"], att_beta = spCLUE_model.train()
        import scipy.sparse as sp

        input_x = adata.X
        if sp.issparse(input_x):
            input_x = input_x.toarray() # 将稀疏矩阵转为普通的 numpy array
        if "raw_count" in graph_dict and sp.issparse(graph_dict["raw_count"]):
            graph_dict["raw_count"] = graph_dict["raw_count"].toarray()
        spCLUE_model = spCLUE.spCLUE(
            input_data=input_x,
            graph_dict=graph_dict,
            n_clusters=n_clusters,
            lambda_ccr=5.0,  # CCR损失权重
            use_zinb=True,  # 是否使用ZINB解码器
            # kappa=0.1,
            beta=0.0,
            gamma=1.0,
        )
        _, adata.obsm["spCLUE"], attention_weights = spCLUE_model.train()
        
        # 5. 聚类
        # 原文件 Cell 10 的逻辑
        refinement = True
        cluster_method = "mclust"
        spCLUE.clustering(
            adata,
            n_clusters,
            key="spCLUE",
            refinement=refinement,
            cluster_methods=cluster_method,
        )
        
        # 6. 计算 ARI
        # 原文件 Cell 12 的逻辑
        # 过滤掉 Ground Truth 为 NA 的区域
        adata_valid = adata[adata.obs.Region.notna()]
        ARI = adjusted_rand_score(adata_valid.obs["Region"], adata_valid.obs["mclust_refined"])
        
        print(f"Sample {sample_name} ARI: {ARI:.8f}")
        ari_results.append(ARI)

        # 绘图：show=False 防止直接显示，便于后续保存
        adata.obs["spCLUE"] = adata.obs["mclust_refined"]
        sc.pl.spatial(
            adata, 
            color=["Region", "spCLUE"], 
            title=["Manual Annotation", f"spCLUE (ARI={round(ARI, 2)})"],
            show=False 
        )
        
        # 保存路径
        save_path = os.path.join(figures_dir, f"{sample_name}.png")
        
        # 保存图片 (bbox_inches='tight' 去除多余白边, dpi=300 保证清晰度)
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        
        # 关闭当前图形，释放内存 (在循环中非常重要，否则内存会爆)
        plt.close()
        
        print(f"Figure saved to: {save_path}")
        
    except Exception as e:
        print(f"Error processing sample {sample_name}: {e}")

# 7. 输出最终统计结果
print(f"\n{'='*20} Final Results {'='*20}")
if ari_results:
    mean_ari = np.mean(ari_results)
    median_ari = np.median(ari_results)
    print(f"ARI per slice: {[round(x, 5) for x in ari_results]}")
    print(f"Mean ARI: {mean_ari:.4f}")
    print(f"Median ARI: {median_ari:.4f}")
else:
    print("No ARI results collected.")

✅ 成功强制加载 libR.so
R 环境路径: /home/pxy/miniconda3/envs/r40/lib/R
Start processing 12 slices...

==================== Processing Sample: 151507 ====================
Preprocessing starting according to CSMVL description...
Selecting top 3000 HVGs...
Normalizing total counts to 10,000...
Scaling and clipping at threshold 10...
Training Start (Enhanced 3-View with CCR) =========================>


 41%|████      | 103/250 [00:04<00:05, 27.45it/s]

epoch 100: ARI=-0.0051, CCR=0.5005, CLU=2.5652, REC=0.5951
tensor([[ 0.9661,  1.0288,  1.1260,  ...,  8.3486,  8.4085,  9.0176],
        [ 0.9916,  1.0300,  1.1917,  ...,  9.1652,  9.0829,  9.0950],
        [ 0.8940,  1.0320,  1.0143,  ...,  9.2618,  9.1654,  9.0550],
        ...,
        [ 0.8959,  1.0314,  1.0125,  ...,  8.7459,  8.5852,  8.4536],
        [ 0.9304,  1.0321,  1.0353,  ...,  5.3457,  5.4967,  5.9842],
        [ 0.8912,  1.0337,  1.0167,  ..., 10.0231, 10.0118,  9.9618]],
       device='cuda:0', grad_fn=<ClampBackward1>)


 82%|████████▏ | 205/250 [00:08<00:01, 28.09it/s]

epoch 200: ARI=-0.0056, CCR=0.5000, CLU=2.5650, REC=0.5868
tensor([[ 0.8499,  1.0606,  1.0244,  ...,  9.2240,  9.3878,  9.4717],
        [ 1.0445,  1.0965,  1.2705,  ...,  8.9307,  8.9004,  8.9295],
        [ 0.8408,  1.0638,  1.0245,  ..., 10.4667, 10.4790, 10.4517],
        ...,
        [ 0.8443,  1.0622,  1.0239,  ...,  9.8716,  9.8489,  9.8191],
        [ 0.8651,  1.0541,  1.0222,  ...,  7.2804,  7.3701,  7.4604],
        [ 0.8389,  1.0648,  1.0249,  ..., 10.8084, 10.8545, 10.8323]],
       device='cuda:0', grad_fn=<ClampBackward1>)


...

epoch 100: ARI=-0.0162, CCR=0.4276, CLU=2.5652, REC=0.7076
tensor([[1.1751, 1.1318, 0.9175,  ..., 1.0334, 1.1230, 1.3928],
        [1.1677, 1.1276, 0.9203,  ..., 1.0336, 1.1179, 1.3749],
        [1.0787, 1.0644, 1.3704,  ..., 0.9597, 1.0843, 1.9793],
        ...,
        [1.0795, 1.0647, 1.3732,  ..., 0.9595, 1.0852, 1.9874],
        [1.0811, 1.0644, 1.3744,  ..., 0.9583, 1.0866, 1.9912],
        [1.1594, 1.1226, 0.9224,  ..., 1.0330, 1.1130, 1.3578]],
       device='cuda:0', grad_fn=<ClampBackward1>)


 82%|████████▏ | 204/250 [00:06<00:01, 32.64it/s]

epoch 200: ARI=0.0034, CCR=0.4218, CLU=2.5650, REC=0.6998
tensor([[1.1896, 1.2053, 0.9183,  ..., 1.0913, 1.1620, 1.4138],
        [1.1802, 1.1953, 0.9218,  ..., 1.0871, 1.1542, 1.3942],
        [1.0984, 1.0831, 1.4011,  ..., 1.0206, 1.1567, 1.8967],
        ...,
        [1.0987, 1.0834, 1.4022,  ..., 1.0207, 1.1572, 1.8994],
        [1.1020, 1.0864, 1.4158,  ..., 1.0216, 1.1629, 1.9318],
        [1.1722, 1.1869, 0.9248,  ..., 1.0834, 1.1476, 1.3778]],
       device='cuda:0', grad_fn=<ClampBackward1>)


100%|██████████| 250/250 [00:07<00:00, 33.00it/s]


Training Finished =================<
fitting ...
  |======================================================================| 100%
Sample 151676 ARI: 0.46436555
Figure saved to: figures_test/151676.png

==================== Final Results ====================
ARI per slice: [0.35481, 0.27147, 0.39704, 0.57069, 0.6254, 0.42535, 0.56986, 0.68482, 0.44046, 0.42567, 0.28947, 0.46437]
Mean ARI: 0.4600
Median ARI: 0.4331
